# Multi-task Reward Router Training (Utility + Task Type)

This notebook trains a multi-task VLM router that predicts:
1. **Task Type**: Classification head (which router task is this?)
2. **Utility-based Reward**: Regression head (predicting utility for `accuracy`, `cheap`, `fast`, `balanced` modes).

It connects to the Postgres DB, loads joined data (`vlm_samples` + `vlm_responses`), builds a multi-task dataset, extends the `RewardRouterModel`, and trains it end-to-end.

## 1. Setup & Imports

In [ ]:
import os
import sys
import logging
from pathlib import Path

# Set project root
PROJECT_ROOT = "/Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Which_VLM_Router/artemis_final/router_train"
os.chdir(PROJECT_ROOT)
sys.path.append(PROJECT_ROOT)

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("multitask_router")

print(f"Project root set to: {PROJECT_ROOT}")

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import json
import sqlalchemy
from dataclasses import asdict, dataclass
from typing import Dict, List, Optional, Tuple, Any
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer
from scipy.stats import pearsonr

# Local imports
try:
    from config import Config
    from training import dataset as ds
    from training import train_reward_router
    from training import train_utils
    from models import reward_router
except ImportError as e:
    print(f"Error importing local modules: {e}")
    print("Make sure you are running this notebook from the notebook directory and PROJECT_ROOT is correct.")

# Set plot style
sns.set_theme(style="whitegrid")
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

## 2. Load Config & DB Connection

In [ ]:
cfg = Config.default()
print("Loaded configuration:")
print(cfg)

# Create DB connection
db_url = cfg.db.get_connection_string()
engine = sqlalchemy.create_engine(db_url)

# Verify connection
try:
    with engine.connect() as conn:
        result = conn.execute(sqlalchemy.text("SELECT 1"))
        print("\nDatabase connection successful! \u2705")
        
        # Check for required tables
        inspector = sqlalchemy.inspect(conn)
        tables = inspector.get_table_names()
        required_tables = ["vlm_samples", "vlm_responses", "vlm_evaluations"]
        missing = [t for t in required_tables if t not in tables]
        
        if missing:
            print(f"\u26a0\ufe0f Warning: Missing tables: {missing}")
        else:
            print(f"All required tables present: {required_tables} \u2705")
            
except Exception as e:
    print(f"\n\u274c Database connection failed: {e}")
    # Stop execution if DB fails (comment out if debugging offline)
    # raise e

## 3. Load Joined Data with Utility Columns

In [ ]:
def load_profiles_data(engine):
    """Load joined data from vlm_samples, vlm_responses, and vlm_evaluations."""
    
    query = """
    SELECT 
        s.sample_id,
        s.router_task,
        s.data_split,
        s.prompt_text as prompt_raw,
        s.image_id,
        s.prompt_len_words,
        s.img_width,
        s.img_height,
        s.img_aspect_ratio,

        r.model_name,
        r.ok,
        r.confidence_score,
        r.estimated_cost_usd,
        r.latency_ms,
        r.input_tokens,
        r.output_tokens,
        
        -- Utility Columns (computed in DB)
        r.utility_accuracy,
        r.utility_cheap,
        r.utility_fast,
        r.utility_balanced,

        -- Evaluation Metrics (optional, for reference)
        e.glider_score,
        e.semantic_f1_f1,
        e.judge_molmo_score

    FROM vlm_responses r
    JOIN vlm_samples s ON r.sample_id = s.sample_id
    LEFT JOIN vlm_evaluations e ON r.response_id = e.response_id
    WHERE r.ok = true
    """
    
    print("Executing SQL query...")
    df = pd.read_sql(query, engine)
    print(f"Loaded {len(df)} rows.")
    return df

profiles_df = load_profiles_data(engine)

# Check for required columns
required_cols = [
    "sample_id", "router_task", "data_split", "prompt_raw", "model_name",
    "ok", "estimated_cost_usd", "latency_ms", "confidence_score",
    "utility_accuracy", "utility_cheap", "utility_fast", "utility_balanced"
]

missing_cols = [c for c in required_cols if c not in profiles_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}\nEnsure DB has the new utility columns!")

print("All required columns present.")
display(profiles_df.head())

## 4. Clean & Validate Data

In [ ]:
# 1. Drop rows with null utility_accuracy (must have valid accuracy utility to train)
initial_len = len(profiles_df)
profiles_df = profiles_df.dropna(subset=["utility_accuracy"])
print(f"Dropped {initial_len - len(profiles_df)} rows with null utility_accuracy.")

# 2. Fill missing cost/latency with medians per model (if any)
# Typically DB should have these, but safety first
for metric in ["estimated_cost_usd", "latency_ms", "confidence_score"]:
    if profiles_df[metric].isnull().any():
        print(f"Filling missing {metric} with model medians...")
        profiles_df[metric] = profiles_df.groupby("model_name")[metric].transform(
            lambda x: x.fillna(x.median())
        )

# 3. Deduplicate (sample_id, model_name)
profiles_df = profiles_df.drop_duplicates(subset=["sample_id", "model_name"])
print(f"Final clean dataset size: {len(profiles_df)} rows")

# 4. Stats
print("\nTask Distribution:")
print(profiles_df["router_task"].value_counts())

print("\nModel Distribution:")
print(profiles_df["model_name"].value_counts())

## 5. Build Training Target Table

In [ ]:
MODES = ["accuracy", "cheap", "fast", "balanced"]

rows = []

for _, row in tqdm(profiles_df.iterrows(), total=len(profiles_df), desc="Building target table"):
    base_data = {
        "sample_id": row["sample_id"],
        "model_name": row["model_name"],
        "router_task": row["router_task"],
        "data_split": row["data_split"],
        "prompt_raw": row["prompt_raw"],
        "prompt_len_words": row["prompt_len_words"],
        "img_width": row["img_width"],
        "img_height": row["img_height"],
        "img_aspect_ratio": row["img_aspect_ratio"],
        "estimated_cost_usd": row["estimated_cost_usd"],
        "latency_ms": row["latency_ms"],
    }
    
    # Create a row for each mode
    # Accuracy
    if pd.notnull(row["utility_accuracy"]):
        rows.append({**base_data, "mode_name": "accuracy", "reward": row["utility_accuracy"]})
        
    # Cheap
    if pd.notnull(row["utility_cheap"]):
        rows.append({**base_data, "mode_name": "cheap", "reward": row["utility_cheap"]})
        
    # Fast
    if pd.notnull(row["utility_fast"]):
        rows.append({**base_data, "mode_name": "fast", "reward": row["utility_fast"]})
        
    # Balanced
    if pd.notnull(row["utility_balanced"]):
        rows.append({**base_data, "mode_name": "balanced", "reward": row["utility_balanced"]})

reward_df = pd.DataFrame(rows)
print(f"Created reward dataframe with {len(reward_df)} rows (expanded by modes).")
display(reward_df.head())

## 6. Create ID Mappings

In [ ]:
model_names = sorted(reward_df["model_name"].unique())
mode_names = MODES
task_names = sorted(reward_df["router_task"].unique())

print(f"Models ({len(model_names)}): {model_names}")
print(f"Modes ({len(mode_names)}): {mode_names}")
print(f"Tasks ({len(task_names)}): {task_names}")

model_to_id = {m: i for i, m in enumerate(model_names)}
mode_to_id = {m: i for i, m in enumerate(mode_names)}
task_to_id = {t: i for i, t in enumerate(task_names)}

reward_df["model_id"] = reward_df["model_name"].map(model_to_id)
reward_df["mode_id"] = reward_df["mode_name"].map(mode_to_id)
reward_df["task_id"] = reward_df["router_task"].map(task_to_id)

# Save mappings
os.makedirs("data", exist_ok=True)
with open("data/model_index.json", "w") as f: json.dump(model_names, f)
with open("data/mode_index.json", "w") as f: json.dump(mode_names, f)
with open("data/task_index.json", "w") as f: json.dump(task_names, f)

# Save processed dataset
reward_df.to_parquet("data/router_reward_multitask_dataset.parquet", index=False)
print("Saved dataset and indices to data/")

## 7. Train/Val/Test Split

In [ ]:
if "data_split" in reward_df.columns and reward_df["data_split"].notnull().all():
    print("Using existing data_split column.")
    # Standardize names just in case
    # Should be 'train', 'val', 'test'
    print("Split counts:")
    print(reward_df["data_split"].value_counts())
else:
    print("Creating new random split by sample_id (80/10/10)...")
    unique_samples = reward_df["sample_id"].unique()
    np.random.seed(42)
    np.random.shuffle(unique_samples)
    
    n = len(unique_samples)
    train_n = int(0.80 * n)
    val_n = int(0.10 * n)
    
    train_ids = set(unique_samples[:train_n])
    val_ids = set(unique_samples[train_n:train_n+val_n])
    test_ids = set(unique_samples[train_n+val_n:])
    
    def assign_split(sid):
        if sid in train_ids: return "train"
        if sid in val_ids: return "val"
        return "test"
    
    reward_df["data_split"] = reward_df["sample_id"].map(assign_split)
    print("New split counts:")
    print(reward_df["data_split"].value_counts())

## 8. Dataset & Dataloaders\nWe extend `RewardRouterDataset` to include `task_id`.

In [ ]:
class MultiTaskRewardRouterDataset(ds.RewardRouterDataset):
    """Extended dataset that includes task_id."""
    
    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        # Get base items (input_ids, attention_mask, model_id, mode_id, reward, sample_id)
        data = super().__getitem__(idx)
        
        # Create task_id tensor
        row = self.df.iloc[idx]
        data["task_id"] = torch.tensor(row["task_id"], dtype=torch.long)
        
        return data

def multitask_collate_fn(batch: List[Dict]) -> Dict[str, torch.Tensor]:
    """Collate with task_id."""
    # Helper to stack tensors
    base_batch = ds.collate_fn(batch)
    
    # Add task_id
    task_id = torch.stack([item["task_id"] for item in batch])
    base_batch["task_id"] = task_id
    
    return base_batch

# Override dataset class in ds module temporarily or use custom build function
# Here we implement a custom build_dataloaders for clarity

def build_multitask_dataloaders(
    df: pd.DataFrame,
    tokenizer: AutoTokenizer,
    config,
    max_length: int = 256
):
    # Reuse the split logic from ds.split_by_data_split_column (since we added 'data_split' col)
    train_df = df[df["data_split"] == "train"].copy()
    val_df = df[df["data_split"] == "val"].copy()
    test_df = df[df["data_split"] == "test"].copy()
    
    # Create datasets
    train_ds = MultiTaskRewardRouterDataset(
        train_df, tokenizer, max_seq_length=max_length, split="train", 
        enable_augmentation=config.enable_metadata_augmentation,
        question_only_ratio=config.question_only_ratio,
        image_metadata_only_ratio=config.image_metadata_only_ratio,
        full_metadata_ratio=config.full_metadata_ratio
    )
    val_ds = MultiTaskRewardRouterDataset(val_df, tokenizer, max_seq_length=max_length, split="val", enable_augmentation=False)
    test_ds = MultiTaskRewardRouterDataset(test_df, tokenizer, max_seq_length=max_length, split="test", enable_augmentation=False)
    
    # Create loaders
    train_loader = torch.utils.data.DataLoader(
        train_ds,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=config.num_workers,
        collate_fn=multitask_collate_fn
    )
    val_loader = torch.utils.data.DataLoader(
        val_ds,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=config.num_workers,
        collate_fn=multitask_collate_fn
    )
    test_loader = torch.utils.data.DataLoader(
        test_ds,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=config.num_workers,
        collate_fn=multitask_collate_fn
    )
    
    return train_loader, val_loader, test_loader

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained(cfg.model.text_encoder_name)

# Create dataloaders
train_loader, val_loader, test_loader = build_multitask_dataloaders(
    reward_df,
    tokenizer,
    cfg.training,
    max_length=cfg.model.max_seq_len
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

# Test check
batch = next(iter(train_loader))
print("Batch keys:", batch.keys())
print("Task ID shape:", batch["task_id"].shape)

## 9. Extend Reward Router Model (Task Head)

In [ ]:
class MultiTaskRewardRouterModel(reward_router.RewardRouterModel):
    """Reward Router model with an additional task classification head."""
    
    def __init__(self, config, num_models, num_modes, num_tasks, text_encoder_hidden_size=None):
        super().__init__(config, num_models, num_modes, text_encoder_hidden_size)
        
        self.num_tasks = num_tasks
        
        # Task classification head
        # Takes text representation as input
        self.task_head = nn.Linear(self.text_hidden_size, num_tasks)
        
        # Initialize
        nn.init.xavier_uniform_(self.task_head.weight)
        nn.init.zeros_(self.task_head.bias)
        
        logger.info(f"Initialized task head for {num_tasks} tasks.")
        
    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        model_id: torch.Tensor,
        mode_id: torch.Tensor,
    ) -> Dict[str, torch.Tensor]:
        # Encode text (reuse parent logic or call text_encoder directly)
        text_outputs = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        
        if self.config.text_pooling == "cls":
            h_text = text_outputs.last_hidden_state[:, 0, :]
        else:
            # Mean pooling
            token_embeddings = text_outputs.last_hidden_state
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
            h_text = torch.sum(token_embeddings * input_mask_expanded, dim=1) / torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)

        # Task logits (from text only)
        task_logits = self.task_head(h_text)
        
        # Reward prediction (legacy path)
        h_model = self.model_embedding(model_id)
        h_mode = self.mode_embedding(mode_id)
        
        h = torch.cat([h_text, h_model, h_mode], dim=-1)
        h = self.mlp(h)
        reward_hat = self.output(h).squeeze(-1)
        
        return {
            "reward_hat": reward_hat,
            "task_logits": task_logits
        }

# Create model
num_models = len(model_names)
num_modes = len(mode_names)
num_tasks = len(task_names)

model = MultiTaskRewardRouterModel(
    config=cfg.model,
    num_models=num_models,
    num_modes=num_modes,
    num_tasks=num_tasks
)

device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
model.to(device)
print(f"Model created and moved to {device}")

## 10. Multi-task Training Loop

In [ ]:
class MultiTaskTrainer(train_reward_router.RewardRouterTrainer):
    """Extends trainer to handle multi-task loss."""
    
    def __init__(self, *args, lambda_task=0.1, **kwargs):
        super().__init__(*args, **kwargs)
        self.lambda_task = lambda_task
        self.task_criterion = nn.CrossEntropyLoss()
        
    def train_epoch(self) -> Dict[str, float]:
        self.model.train()
        total_loss = 0.0
        total_routing_loss = 0.0
        total_task_loss = 0.0
        total_task_acc = 0.0
        
        all_preds = []
        all_targets = []
        
        pbar = tqdm(self.train_loader, desc=f"Epoch {self.current_epoch + 1} [Train]")
        
        for batch in pbar:
            # Move to device
            input_ids = batch["input_ids"].to(self.device)
            attention_mask = batch["attention_mask"].to(self.device)
            model_id = batch["model_id"].to(self.device)
            mode_id = batch["mode_id"].to(self.device)
            reward = batch["reward"].to(self.device)
            task_id = batch["task_id"].to(self.device)
            
            # Forward
            outputs = self.model(
                input_ids=input_ids, 
                attention_mask=attention_mask, 
                model_id=model_id, 
                mode_id=mode_id
            )
            
            reward_hat = outputs["reward_hat"]
            task_logits = outputs["task_logits"]
            
            # Losses
            loss_routing = self.criterion(reward_hat, reward)
            loss_task = self.task_criterion(task_logits, task_id)
            
            loss = loss_routing + self.lambda_task * loss_task
            
            # Backward
            self.optimizer.zero_grad()
            loss.backward()
            
            if self.train_config.gradient_clip_norm > 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.train_config.gradient_clip_norm)
                
            self.optimizer.step()
            if self.scheduler:
                self.scheduler.step()
                
            # Metrics
            total_loss += loss.item()
            total_routing_loss += loss_routing.item()
            total_task_loss += loss_task.item()
            
            task_pred = task_logits.argmax(dim=-1)
            task_acc = (task_pred == task_id).float().mean()
            total_task_acc += task_acc.item()
            
            all_preds.extend(reward_hat.detach().cpu().numpy())
            all_targets.extend(reward.detach().cpu().numpy())
            
            pbar.set_postfix({"loss": f"{loss.item():.3f}", "t_acc": f"{task_acc.item():.2f}"})
            
        # Aggregates
        n = len(self.train_loader)
        try: corr, _ = pearsonr(all_preds, all_targets)
        except: corr = 0.0
            
        return {
            "loss": total_loss / n,
            "loss_routing": total_routing_loss / n,
            "loss_task": total_task_loss / n,
            "task_acc": total_task_acc / n,
            "corr": corr,
            "lr": self.optimizer.param_groups[0]["lr"]
        }

    @torch.no_grad()
    def validate(self) -> Dict[str, float]:
        self.model.eval()
        total_loss = 0.0
        total_task_acc = 0.0
        all_preds, all_targets = [], []
        
        pbar = tqdm(self.val_loader, desc=f"Epoch {self.current_epoch + 1} [Val]")
        
        for batch in pbar:
            input_ids = batch["input_ids"].to(self.device)
            attention_mask = batch["attention_mask"].to(self.device)
            model_id = batch["model_id"].to(self.device)
            mode_id = batch["mode_id"].to(self.device)
            reward = batch["reward"].to(self.device)
            task_id = batch["task_id"].to(self.device)
            
            outputs = self.model(input_ids, attention_mask, model_id, mode_id)
            reward_hat = outputs["reward_hat"]
            task_logits = outputs["task_logits"]
            
            loss_routing = self.criterion(reward_hat, reward)
            loss_task = self.task_criterion(task_logits, task_id)
            loss = loss_routing + self.lambda_task * loss_task
            
            total_loss += loss.item()
            
            task_pred = task_logits.argmax(dim=-1)
            total_task_acc += (task_pred == task_id).float().mean().item()
            
            all_preds.extend(reward_hat.cpu().numpy())
            all_targets.extend(reward.cpu().numpy())
            
        n = len(self.val_loader)
        try: corr, _ = pearsonr(all_preds, all_targets)
        except: corr = 0.0
            
        return {
            "loss": total_loss / n,
            "task_acc": total_task_acc / n,
            "corr": corr
        }

# Init trainer
trainer = MultiTaskTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=cfg,
    device=cfg.training.device,
    lambda_task=0.1
)

# Train
history = trainer.train()

## 11. Evaluation: Routing Accuracy vs Oracle

In [ ]:
# Load best model
best_path = cfg.paths.get_checkpoint_path()
if os.path.exists(best_path):
    print(f"Loading best model from {best_path}")
    checkpoint = torch.load(best_path, map_location=device)
    # Note: we need to use our MultiTask class to load, but the checkpoint keys might match
    # We just load state_dict here since structure is compatible
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
else:
    print("No best model found, using current model state.")
    
# Compute Routing Accuracy
# We need to iterate test set, predict reward for ALL models for each sample, select max, compare with oracle

# 1. Group test data by sample
test_samples = test_df.groupby("sample_id")
results = []

print("Evaluating routing accuracy...")

model.eval()
with torch.no_grad():
    # Iterate random subset of samples to save time if large
    sample_ids = test_df["sample_id"].unique()
    
    for sid in tqdm(sample_ids):
        group = test_df[test_df["sample_id"] == sid]
        
        # Get raw prompt from first row
        prompt_raw = group.iloc[0]["prompt_raw"]
        # For now, we assume simple prompt-based input building
        # Ideally we reuse dataset._build_input_text logic but let's mock the input for now 
        # or reuse a single row from the group to get metadata
        
        # We need to evaluate each mode
        for mode_name in MODES:
            mode_idx = mode_to_id[mode_name]
            
            # Filter group for this mode to get Ground Truth rewards
            # But wait: available models might differ per sample. 
            # We route among AVAILABLE models for this sample.
            # In offline eval, we only know the ground truth for models we HAVE responses for.
            sub_group = group[group["mode_name"] == mode_name]
            if len(sub_group) < 2:
                continue # Need at least 2 models to route
                
            # Prepare batch input for all available models
            # We replicate the input text N times
            # Creating a mini-batch of [N_models]
            
            # HACK: construct input text using dataset logic by making a temporary DF row
            # We assume the dataset class logic is consistent. 
            # To be efficient, we can't easily use the dataset class here without refactoring.
            # Let's manually tokenize the prompt + metadata from the first row of sub_group
            row = sub_group.iloc[0]
            # Reconstruct simplified input text
            input_text = f"[ROUTER] PromptLenWords: {row['prompt_len_words']}. ImgWidth: {row['img_width']}. ImgHeight: {row['img_height']}. ImgAR: {row['img_aspect_ratio']:.2f}. Question: {row['prompt_raw']}"
            
            encoding = tokenizer(
                input_text, 
                max_length=cfg.model.max_seq_len, 
                padding="max_length", 
                truncation=True, 
                return_tensors="pt"
            )
            
            n_cands = len(sub_group)
            input_ids = encoding["input_ids"].repeat(n_cands, 1).to(device)
            att_mask = encoding["attention_mask"].repeat(n_cands, 1).to(device)
            mode_ids = torch.tensor([mode_idx] * n_cands, device=device)
            
            # Candidate model IDs
            cand_model_names = sub_group["model_name"].tolist()
            cand_model_ids = torch.tensor([model_to_id[m] for m in cand_model_names], device=device)
            
            # Forward
            outputs = model(input_ids, att_mask, cand_model_ids, mode_ids)
            rewards_hat = outputs["reward_hat"]
            
            # Decision
            best_idx = rewards_hat.argmax().item()
            chosen_model = cand_model_names[best_idx]
            
            # Oracle
            oracle_idx = sub_group["reward"].argmax()
            oracle_model = sub_group.iloc[oracle_idx]["model_name"]
            oracle_reward = sub_group.iloc[oracle_idx]["reward"]
            
            # Router Reward (Actual utility of chosen model)
            router_reward = sub_group.iloc[best_idx]["reward"]
            
            # Baselines
            # Random
            rand_idx = np.random.randint(n_cands)
            rand_reward = sub_group.iloc[rand_idx]["reward"]
            
            results.append({
                "sample_id": sid,
                "mode": mode_name,
                "is_optimal": (chosen_model == oracle_model),
                "oracle_reward": oracle_reward,
                "router_reward": router_reward,
                "random_reward": rand_reward
            })

eval_df = pd.DataFrame(results)
print("Evaluation complete.")
display(eval_df.head())

# Summary
summary = eval_df.groupby("mode").agg({
    "is_optimal": "mean",
    "oracle_reward": "mean",
    "router_reward": "mean",
    "random_reward": "mean"
}).reset_index()

summary["rel_performance"] = summary["router_reward"] / summary["oracle_reward"]
display(summary)

## 12. Save Artifacts & Wrap Up

In [ ]:
# Save eval summary
os.makedirs("results", exist_ok=True)
eval_df.to_csv("results/multitask_eval_details.csv", index=False)
summary.to_csv("results/multitask_eval_summary.csv", index=False)

# Save training plots
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.plot(history["train_loss"], label="Train")
plt.plot(history["val_loss"], label="Val")
plt.title("Loss")
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(history["train_corr"], label="Train")
plt.plot(history["val_corr"], label="Val")
plt.title("Reward Correlation")
plt.legend()

plt.subplot(1, 3, 3)
# Note: we didn't save task_acc in the history dict returned by base trainer structure by default
# unless we modified the Trainer.train() method to return custom dictionary keys.
# But let's check if our CustomTrainer updated the history dict in place? 
# Actually the 'history' logic in base trainer only appends pre-defined keys.
# Since we didn't override train(), only train_epoch(), the history keys for task_acc naturally won't exist.
# That's fine, we focus on loss and corr which are the main ones.
pass 

plt.tight_layout()
plt.savefig("results/multitask_training_curves.png")
plt.show()

print("Saved artifacts to results/")